# 01 -- SuperStore Data Extraction and Initial Inspection

**Project:** SuperStore Retail Performance & Profitability Analysis  
**Workflow:** Tableau-first notebook pipeline rebuilt from the raw dataset  
**Goal:** Validate the raw source before cleaning and keep the Tableau workflow isolated from the Google Sheets and Looker Studio assets.

## 1. Environment Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "SuperStore_Analysis/Tableau_Analysis/scripts/superstore_pipeline.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root from the notebook environment.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from SuperStore_Analysis.Tableau_Analysis.scripts.superstore_pipeline import (
    FINAL_SCHEMA,
    LOYAL_CUSTOMER_MIN_PURCHASES,
    build_clean_dataset,
    build_tableau_ready_dataset,
    compute_project_metrics,
    copy_raw_snapshot,
    export_pipeline_outputs,
    load_existing_cleaned_dataset,
    load_raw_dataset,
    prepare_analysis_frame,
    resolve_project_paths,
    validate_final_schema,
)

PATHS = resolve_project_paths(REPO_ROOT / "SuperStore_Analysis" / "Tableau_Analysis")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


## 2. Load the Raw Source and Reference Benchmark

The Tableau workflow works from the raw project dataset and keeps a local raw snapshot inside `Tableau_Analysis/data/raw/`. The Google Sheets cleaned file is loaded only as a reference benchmark for schema checks.

In [2]:
snapshot_path = copy_raw_snapshot(PATHS)
df_raw = load_raw_dataset(PATHS)
df_benchmark = load_existing_cleaned_dataset(PATHS)

summary = pd.DataFrame({
    "Dataset": ["Raw source", "Reference cleaned benchmark"],
    "Rows": [len(df_raw), len(df_benchmark)],
    "Columns": [df_raw.shape[1], df_benchmark.shape[1]],
    "Path": [str(snapshot_path), str(PATHS.source_cleaned_path)],
})
display(summary)


,Dataset,Rows,Columns,Path
0,Raw source,9994,21,/Users/ravleensingh/Documents/Capstone/DVA_Cap...
1,Reference cleaned benchmark,9993,37,/Users/ravleensingh/Documents/Capstone/DVA_Cap...


## 3. Raw Column Layout

In [3]:
raw_columns = pd.DataFrame({
    "Column Position": range(1, len(df_raw.columns) + 1),
    "Raw Column Name": df_raw.columns,
})
display(raw_columns)


,Column Position,Raw Column Name
0,1,Row ID
1,2,Order ID
2,3,Order Date
3,4,Ship Date
4,5,Ship Mode
5,6,Customer ID
6,7,Customer Name
7,8,Segment
8,9,Country
9,10,City


## 4. Dataset Preview

In [4]:
display(df_raw.head(8))
display(df_raw.tail(5))


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52
5,6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.86,7,0.00,14.17
6,7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.00,1.97
7,8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.15,6,0.20,90.72


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
9989,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,Florida,33180,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.25,3,0.20,4.10
9990,9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.96,2,0.00,15.63
9991,9992,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.58,2,0.20,19.39
9992,9993,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.60,4,0.00,13.32
9993,9994,CA-2017-119914,2017-05-04,2017-05-09,Second Class,CC-12220,Chris Cortes,Consumer,United States,Westminster,California,92683,West,OFF-AP-10002684,Office Supplies,Appliances,"Acco 7-Outlet Masterpiece Power Center, Wihtou...",243.16,2,0.00,72.95


## 5. Data Quality and Business Coverage

In [5]:
df_dates = df_raw.copy()
df_dates["Order Date"] = pd.to_datetime(df_dates["Order Date"])
df_dates["Ship Date"] = pd.to_datetime(df_dates["Ship Date"])

quality_summary = pd.DataFrame({
    "Check": [
        "Row count",
        "Column count",
        "Missing values",
        "Duplicate full rows",
        "Order date start",
        "Order date end",
        "Unique orders",
        "Unique customers",
        "Unique products",
        "States covered",
    ],
    "Value": [
        len(df_raw),
        df_raw.shape[1],
        int(df_raw.isna().sum().sum()),
        int(df_raw.duplicated().sum()),
        df_dates["Order Date"].min().date(),
        df_dates["Order Date"].max().date(),
        df_raw["Order ID"].nunique(),
        df_raw["Customer ID"].nunique(),
        df_raw["Product ID"].nunique(),
        df_raw["State"].nunique(),
    ],
})
display(quality_summary)


,Check,Value
0,Row count,9994
1,Column count,21
2,Missing values,0
3,Duplicate full rows,0
4,Order date start,2014-01-03
5,Order date end,2017-12-30
6,Unique orders,5009
7,Unique customers,793
8,Unique products,1862
9,States covered,49


## 6. Duplicate Transaction Detection

The raw source contains one duplicated business row when `Row ID` is excluded. That duplicate is removed in the cleaning notebook to keep the Tableau workflow reproducible and aligned with the published dashboard story.

In [6]:
dedupe_subset = [column for column in df_raw.columns if column != "Row ID"]
duplicate_rows = df_raw.loc[df_raw.duplicated(subset=dedupe_subset, keep=False)].copy()
display(duplicate_rows.sort_values(["Order ID", "Product ID", "Row ID"]))


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
3405,3406,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.37,2,0.30,-12.06
3406,3407,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.37,2,0.30,-12.06


## 7. Final Schema Target

The exported Tableau-ready dataset will contain only the approved 37 columns. Extra helper fields will be created inside Tableau as calculated fields instead of being saved into the CSV.

In [7]:
final_schema_frame = pd.DataFrame({
    "Column Position": range(1, len(FINAL_SCHEMA) + 1),
    "Final Tableau Schema": FINAL_SCHEMA,
})
display(final_schema_frame)


,Column Position,Final Tableau Schema
0,1,Row ID
1,2,Transaction Id (PK)
2,3,Order ID
3,4,Order Date
4,5,Year
5,6,Month
6,7,Quarter
7,8,Ship Date
8,9,Shipping Delay
9,10,Shipping Speed


## 8. Extraction Takeaways

- The raw source is structurally complete and ready for notebook-based cleaning.
- One duplicate business transaction is present and must be removed.
- The Tableau workflow will keep only the final 37 dashboard-facing columns in the exported CSV.
- Existing Google Sheets and Looker Studio project files remain untouched and are used only for reference.